# Lekcija 01 - Uvod v AI agente

Dobrodošli na prvi lekciji v tečaju **AI agenti za začetnike**!

**AI agent** je program, ki uporablja velik jezikovni model (LLM) kot svoj mehanizem sklepanja in lahko izvede *dejanja* v resničnem svetu — kliče API-je, poizveduje po podatkovnih bazah ali izvaja kodo — da doseže cilj v imenu uporabnika.

V tem zvezku boste zgradili svojega prvega agenta: **potovalnega agenta**, ki priporoča počitniške destinacije. Medtem se boste naučili, kako:

1. Povezati se z Microsoft Foundry Agent Service z uporabo **Microsoft Agent Framework**.
2. Agentu dati **orodje** — preprosto Python funkcijo, ki jo lahko kliče.
3. Zagnati agenta in pregledati njegov odgovor.
4. Tokovno prenašati agentov odgovor znak za znakom.


## Namestitev

Pred izvajanjem tega zvezka se prepričajte, da imate:

1. **Microsoft Foundry projekt** z nameščenim klepetalnim modelom (npr. `gpt-4.1-mini`).
2. **Prijavljeni v Azure CLI** — zaženite `az login` v vašem terminalu.
3. **Nastavljene zahtevane okoljske spremenljivke:**
   - `AZURE_AI_PROJECT_ENDPOINT` — končna točka vašega Microsoft Foundry projekta.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašega nameščenega modela.

Celica spodaj namesti potrebne Python pakete.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Ustvarjanje vašega prvega agenta

Agent potrebuje dve stvari:

- **Navodila**, ki mu povedo *kdo je* in *kako naj se obnaša* (sistemski poziv).
- **Orodja** — Python funkcije okrašene z `@tool`, ki jih agent lahko kliče za pridobivanje informacij ali izvajanje dejanj.

Spodaj definiramo enostavno orodje, ki vrne seznam priljubljenih turističnih destinacij. Agent bo uporabljal to orodje, ko uporabnik vpraša za priporočila glede potovanj.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Pretakanje odgovorov

Za bolj interaktivno izkušnjo lahko odgovor agenta **pretakate**. Namesto da čakate na celoten odgovor, agent sproti predvaja tekstovne dele, takoj ko so ustvarjeni. To je še posebej uporabno v klepetalnih vmesnikih, kjer želite izhod prikazovati v realnem času.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Povzetek

V tej lekciji ste se naučili, kako:

- **Ustvariti ponudnika**, ki se poveže s storitvijo Microsoft Foundry Agent preko `FoundryChatClient`.
- **Določiti orodje** z uporabo dekoratorja `@tool`, da agent lahko kliče vaše Python funkcije.
- **Zagnati agenta** z uporabnikovo sporočilom in izpisati njegov odgovor.
- **Pretakati odgovore** za izhod v realnem času.

V naslednji lekciji bomo bolj poglobljeno raziskali agentske ogrodja in se naučili, kako agentom dati močnejša orodja in sposobnosti večstopenjskega sklepanja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
